In [1]:
%load_ext autoreload
%autoreload 2

### Match stations, using station name, lat, lon. The name matching in Raw data meta_data form, in the folder and in the database

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *

In [3]:
# --- Main workflow ---
# --- read data ---
meta_path = '/fern_data/FERNNorth2024_VF/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/all_stations_insert/V3/rows_output/'
os.makedirs(output_folder, exist_ok=True)


# Raw data read
raw_summary_path = os.path.join(output_folder, "Fern_raw_station_variable_summary.csv")
raw_summary = pd.read_csv(raw_summary_path)

### Implement station name for the new inserved station 

In [6]:
engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)


# unique_station_name = raw_summary['station_name'].dropna().unique()
# unique_ids = raw_summary['native_id'].dropna().unique()


# # Loop over unique station_name and native_id pairs
# for station_name, native_id in zip(unique_station_name, unique_ids):
#     query = sa.text("""
#         UPDATE meta_history AS m
#         SET station_name = :station_name
#         FROM meta_station AS s
#         WHERE m.station_name IS NULL
#           AND s.native_id = :native_id
#           AND m.station_id = s.station_id
#           AND s.network_id = 11;
#     """)
#     with engine.begin() as conn:  # begin() ensures commit
#         conn.execute(query, {"station_name": station_name, "native_id": native_id})
#     print(f"Updated station_name for native_id: {native_id} -> {station_name}")

### The inconsistent native_id

In [30]:
# --- your SQL query ---
query = sa.text("""
    SELECT DISTINCT 
        m.station_name, 
        s.native_id, 
        s.mod_user
    FROM meta_history AS m
    JOIN meta_station AS s 
        ON m.station_id = s.station_id
    WHERE s.network_id = 11;
""")

# --- execute and read into a pandas DataFrame ---
with engine.connect() as conn:
    df = pd.read_sql(query, conn)
# df now contains the result

df_multi = df[df.groupby("station_name")["native_id"].transform("nunique") > 1]
df_multi = df_multi.reset_index(drop=True)
df_multi

,station_name,native_id,mod_user
0,BulkleyWx,1113694,crmp
1,BulkleyWx,PGTIS1,tongli1997
2,Canoe Mountain Stn,Canoe,crmp
3,Canoe Mountain Stn,Canoe Mountain Stn,tongli1997
4,Caribou Pass,Caribou Pass,crmp
5,Caribou Pass,CaribouPass,tongli1997
6,CPFWx,1113682,crmp
7,CPFWx,PGTIS3,tongli1997
8,EndakoWx,1159701,crmp
9,EndakoWx,Endako,tongli1997


In [118]:
engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

### Search by native_id
unique_ids = raw_summary['native_id'].dropna().unique()

query = sa.text(f"""
        SELECT 
            m.station_name,
            v.net_var_name, 
            v.unit,
            MIN(o.obs_time) AS earliest_time,
            MAX(o.obs_time) AS latest_time
        FROM obs_raw AS o
        JOIN meta_history AS m ON o.history_id = m.history_id
        JOIN meta_vars AS v ON o.vars_id = v.vars_id
        JOIN meta_station AS s ON m.station_id = s.station_id
        WHERE s.native_id = :native_id
        GROUP BY v.net_var_name, v.unit, m.station_name, m.lat, m.lon
        ORDER BY v.net_var_name;
    """)

results = []


print(f"Total stations to process: {len(unique_ids)}\n")

for i, nid in enumerate(unique_ids, start=1):
    print(f"[{i}/{len(unique_ids)}] Processing native_id: {nid} ...", end=" ")

    try:
        with engine.connect() as conn:
            df = pd.read_sql(query, conn, params={"native_id": nid})
            df['native_id'] = nid
            results.append(df)

        print(f"✔️ {len(df)} variables")
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        continue

# Combine results
summary_updated_db = pd.concat(results, ignore_index=True)

print("\nDone.")
print(f"Total combined rows: {len(summary_updated_db)}")
print(f"Total successful stations: {summary_updated_db['native_id'].nunique()}")

Total stations to process: 35

[1/35] Processing native_id: AtlSch ... ✔️ 7 variables
[2/35] Processing native_id: Barren ... ✔️ 13 variables
[3/35] Processing native_id: Blackhawk ... 

KeyboardInterrupt: 

### search the variable in database by station_name

In [4]:
df = pd.read_csv(output_folder + "Fern_raw_db_station_matched_v1.csv")
df_unique = df[['station_name', 'native_id', 'matched_name']].copy().drop_duplicates()

# Step 1 — Drop rows where matched_name is NaN **only when** another row exists for same station_name
df_unique['is_nan'] = df_unique['matched_name'].isna()

df_unique = df_unique.sort_values(['station_name', 'is_nan'])

# Step 2 — For duplicates of station_name, keep first (the non-NaN one)
df_unique = df_unique.drop_duplicates(subset='station_name', keep='first')

# # Step 3 — Drop helper column & reset index
df_unique = df_unique.drop(columns='is_nan').reset_index(drop=True)

# Fill NaN in matched_name with station_name
df_unique['matched_name'] = df_unique['matched_name'].fillna(df_unique['station_name'])

df_unique


,station_name,native_id,matched_name
0,Atlin School,AtlSch,Atlin School
1,BarrenWx,Barren,BarrenWx
2,BlackhawkWx,Blackhawk,Blackhawk
3,BoulderWx,Boulder Creek,Boulder Creek
4,BowronPit,BowronPit,Bowron Pit
5,BulkleyWx,PGTIS1,BulkleyWx
6,CPFWx,PGTIS3,CPFWx
7,Canoe Mountain Stn,Canoe Mountain Stn,Canoe Mountain Stn
8,Caribou Pass,CaribouPass,Caribou Pass
9,ChapmanWx,Chapman,ChapmanWx


In [7]:
query = sa.text("""
        SELECT 
            m.station_name,
            s.native_id,
            v.net_var_name, 
            v.unit,
            MIN(o.obs_time) AS earliest_time,
            MAX(o.obs_time) AS latest_time
        FROM obs_raw AS o
        JOIN meta_history AS m ON o.history_id = m.history_id
        JOIN meta_vars AS v ON o.vars_id = v.vars_id
        JOIN meta_station AS s ON m.station_id = s.station_id
        WHERE m.station_name LIKE :name
        GROUP BY v.net_var_name, v.unit, m.station_name,s.native_id, m.lat, m.lon
        ORDER BY v.net_var_name;
    """)

results = []

matched_station_name = df_unique['matched_name']

for i, s_name in enumerate(matched_station_name, start=1):
    print(f"[{i}/{len(matched_station_name)}] Processing station_name: {s_name} ...", end=" ")

    try:
        with engine.connect() as conn:
            df = pd.read_sql(query, conn, params={"name": s_name})  # use "name" to match :name
            df['matched_name'] = s_name
            results.append(df)

        print(f"✔️ {len(df)} variables")
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        continue

# Combine results
summary_updated_db_by_name = pd.concat(results, ignore_index=True)
print("\nDone.")
print(f"Total combined rows: {len(summary_updated_db_by_name)}")
print(f"Total successful stations: {summary_updated_db_by_name['station_name'].nunique()}")

[1/35] Processing station_name: Atlin School ... ✔️ 7 variables
[2/35] Processing station_name: BarrenWx ... ✔️ 13 variables
[3/35] Processing station_name: Blackhawk ... ✔️ 12 variables
[4/35] Processing station_name: Boulder Creek ... ✔️ 11 variables
[5/35] Processing station_name: Bowron Pit ... ✔️ 12 variables
[6/35] Processing station_name: BulkleyWx ... ✔️ 20 variables
[7/35] Processing station_name: CPFWx ... ✔️ 20 variables
[8/35] Processing station_name: Canoe Mountain Stn ... ✔️ 18 variables
[9/35] Processing station_name: Caribou Pass ... ✔️ 18 variables
[10/35] Processing station_name: ChapmanWx ... ✔️ 13 variables
[11/35] Processing station_name: Chief Lake ... ✔️ 11 variables
[12/35] Processing station_name: Coalmine Wx ... ✔️ 10 variables
[13/35] Processing station_name: CrookedLk ... ✔️ 9 variables
[14/35] Processing station_name: Crystal Lake ... ✔️ 11 variables
[15/35] Processing station_name: Dennis ... ✔️ 13 variables
[16/35] Processing station_name: Dunster ... ✔️ 

In [23]:
summary_updated_db_by_name = summary_updated_db_by_name.sort_values(by="station_name").reset_index(drop=True)
summary2 = summary_updated_db_by_name.rename(columns={"net_var_name": "db_var_match"})

summary2.to_csv(output_folder + "5.Fern_db_updated_summary.csv", index=False)

summary_updated_db_by_name[
    summary_updated_db_by_name['station_name'] == 'Boulder Creek'
]

,station_name,native_id,net_var_name,unit,earliest_time,latest_time,matched_name
42,Boulder Creek,BoulderCr,WindSpeedms,m s-1,2010-06-07 13:00:00,2023-09-21 14:00:00,Boulder Creek
43,Boulder Creek,BoulderCr,Water_Content_m3_m3_15_cm,m3/m3,2017-05-25 17:00:00,2023-09-21 14:00:00,Boulder Creek
44,Boulder Creek,BoulderCr,TempC,celsius,2010-06-07 13:00:00,2023-09-21 14:00:00,Boulder Creek
45,Boulder Creek,BoulderCr,WindDirection,degree,2010-06-07 13:00:00,2023-09-21 14:00:00,Boulder Creek
46,Boulder Creek,BoulderCr,SolarRadiationWm,W m-2,2010-06-07 13:00:00,2023-09-21 14:00:00,Boulder Creek
47,Boulder Creek,BoulderCr,RH,%,2010-06-07 13:00:00,2023-09-21 14:00:00,Boulder Creek
48,Boulder Creek,BoulderCr,Rainmm,mm,2010-06-07 13:00:00,2023-09-21 14:00:00,Boulder Creek
49,Boulder Creek,BoulderCr,GustSpeedms,m s-1,2010-06-07 13:00:00,2023-09-21 14:00:00,Boulder Creek
50,Boulder Creek,BoulderCr,DewPtC,celsius,2010-06-07 13:00:00,2023-09-21 14:00:00,Boulder Creek
51,Boulder Creek,BoulderCr,Wetness,%,2011-07-13 11:00:00,2023-09-21 14:00:00,Boulder Creek


### Merge the raw station, to obtain the varible and matched varibles with database

In [9]:
import pandas as pd

# Columns to keep
keep_cols = [
    "station_name", "native_id", "station_file_name",
    "variable", "unit_raw",
    "earliest_time_raw", "latest_time_raw",
    "db_var_match"
]

match_cols = [
    "db_var_match"
]

files = [
    "Fern_raw_db_station_matched_v1.csv",
    "Fern_matched_v2_raw_db_station.csv",
    "Fern_manually_matched_v3_raw_db_station.csv"
]

dfs = []

# Read and align columns
for f in files:
    df = pd.read_csv(output_folder + f)
    df = df[[c for c in keep_cols if c in df.columns]]
    dfs.append(df)

# Merge
merged = pd.concat(dfs, ignore_index=True)

# Drop rows with no matching result in any of the match columns
merged = merged.dropna(subset=match_cols, how="all")

# Remove duplicated rows
merged = merged.drop_duplicates()

# Reset index
merged = merged.reset_index(drop=True)

merged = merged.sort_values(by="station_name").reset_index(drop=True)

# Save
merged.to_csv(output_folder + "5.Fern_raw_merged_all_matches.csv", index=False)

print("Final merged shape:", merged.shape)

Final merged shape: (376, 8)


In [19]:
merged

,station_name,native_id,station_file_name,variable,unit_raw,earliest_time_raw,latest_time_raw,db_var_match
0,Atlin School,AtlSch,Atlin school,Rain,mm,2010-08-20 19:00:00,2024-08-10 19:00:00,Rainmm
1,Atlin School,AtlSch,Atlin school,DewPt,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,DewPtC
2,Atlin School,AtlSch,Atlin school,Wind Speed,m/s,2010-08-20 19:00:00,2024-08-10 19:00:00,WindSpeedms
3,Atlin School,AtlSch,Atlin school,Gust Speed,m/s,2010-08-20 19:00:00,2024-08-10 19:00:00,GustSpeedms
4,Atlin School,AtlSch,Atlin school,Wind Direction,ø,2010-08-20 19:00:00,2024-08-10 19:00:00,WindDirection
...,...,...,...,...,...,...,...,...
371,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,RH,%,2007-07-31 15:00:00,2024-10-24 09:00:00,RH
372,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,Temp,°C,2007-07-31 15:00:00,2024-10-24 09:00:00,TempC
373,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,Pressure,mbar,2007-07-31 15:00:00,2024-10-24 09:00:00,Pressurembar
374,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,Rain,mm,2007-07-31 15:00:00,2024-10-24 09:00:00,Rainmm


In [22]:
summary2

,station_name,native_id,db_var_match,earliest_time,latest_time
0,Atlin School,AtlSch,DewPtC,2011-07-28 09:00:00,2024-08-10 19:00:00
1,Atlin School,AtlSch,GustSpeedms,2010-08-20 19:00:00,2024-08-10 19:00:00
2,Atlin School,AtlSch,Rainmm,2010-08-20 20:00:00,2024-08-10 19:00:00
3,Atlin School,AtlSch,RH,2011-07-28 09:00:00,2024-08-10 19:00:00
4,Atlin School,AtlSch,SolarRadiationWm,2010-08-20 19:00:00,2024-08-10 19:00:00
...,...,...,...,...,...
519,Willow-BowronWx,PGTIS2,RH,2023-10-17 10:00:00,2024-10-24 09:00:00
520,Willow-BowronWx,1095439,Rainmm,2007-07-31 15:00:00,2023-10-17 09:00:00
521,Willow-BowronWx,1095439,Wetness,2008-06-12 13:00:00,2023-10-17 09:00:00
522,Willow-BowronWx,1095439,GustSpeedms,2007-07-31 15:00:00,2023-10-17 09:00:00


In [ ]:
summary2 = summary_updated_db_by_name.rename(columns={"net_var_name": "db_var_match"})


summary2 = summary2[['station_name', 'native_id','db_var_match','earliest_time','latest_time']]
merged2  = merged[['station_name','native_id','variable','db_var_match','earliest_time_raw','latest_time_raw']]

final = pd.merge(
    merged2,
    summary2,
    how="left",
    on=["native_id", "db_var_match"]
)


# Assume `final` is your merged DataFrame
# Rename the DB time columns
final = final.rename(columns={
    'earliest_time': 'earliest_time_db',
    'latest_time': 'latest_time_db'
})

# Make sure the time columns are datetime
final['earliest_time_raw'] = pd.to_datetime(final['earliest_time_raw'])
final['latest_time_raw'] = pd.to_datetime(final['latest_time_raw'])
final['earliest_time_db'] = pd.to_datetime(final['earliest_time_db'])
final['latest_time_db'] = pd.to_datetime(final['latest_time_db'])

# Compute differences in days
final['earliest_diff_days'] = (final['earliest_time_raw'] - final['earliest_time_db']).dt.days
final['latest_diff_days'] = (final['latest_time_raw'] - final['latest_time_db']).dt.days

# Display result
print(final.head())

final.to_csv(output_folder + "5.double_check_Fern_raw_db_matches.csv", index=False)

print("Final merged shape:", merged.shape)


  station_name_x native_id        variable   db_var_match   earliest_time_raw  \
0   Atlin School    AtlSch            Rain         Rainmm 2010-08-20 19:00:00   
1   Atlin School    AtlSch           DewPt         DewPtC 2010-08-20 19:00:00   
2   Atlin School    AtlSch      Wind Speed    WindSpeedms 2010-08-20 19:00:00   
3   Atlin School    AtlSch      Gust Speed    GustSpeedms 2010-08-20 19:00:00   
4   Atlin School    AtlSch  Wind Direction  WindDirection 2010-08-20 19:00:00   

      latest_time_raw station_name_y    earliest_time_db      latest_time_db  \
0 2024-08-10 19:00:00   Atlin School 2010-08-20 20:00:00 2024-08-10 19:00:00   
1 2024-08-10 19:00:00   Atlin School 2011-07-28 09:00:00 2024-08-10 19:00:00   
2 2024-08-10 19:00:00   Atlin School 2010-08-20 19:00:00 2024-08-10 19:00:00   
3 2024-08-10 19:00:00   Atlin School 2010-08-20 19:00:00 2024-08-10 19:00:00   
4 2024-08-10 19:00:00   Atlin School 2010-08-20 19:00:00 2024-08-10 19:00:00   

   earliest_diff_days  latest_di